<a href="https://colab.research.google.com/github/shaktidharreddy/Agentic_AI_eduhubspot_aug_sep_2026/blob/main/StrOutputParser_StructuredOutput.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install langchain_google_genai langchain_core dotenv google-colab --upgrade

INFO: pip is looking at multiple versions of google-genai to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.0 MB/s eta 0:00:00
  Using cached langchain_google_genai-4.4.0-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is still looking at multiple versions of google-genai to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.6 MB/s eta 0:00:00
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5

In [2]:
import os
from google.colab import userdata

# Retrieve the API key from Colab secrets and set as environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [3]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash-lite", temperature = 0.7)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a specialized Medical Data Extraction Assistant for MedPulse Analytics.
        Your task is to parse unstructured clinical narratives into a precise JSON format.

        ### Extraction Rules:
        1. **Patient Info**: Identify name, age (as integer), and gender.
        2. **Encounter**: Extract the date of service, the provider name, and a list of reported symptoms.
        3. **Clinical Data**: Identify the primary diagnosis and its corresponding ICD-10 code.
        4. **Medications**: For every medication, extract the name, dosage, and frequency into a list.
        5. **Billing**: Capture the specific Insurance Claim ID and all CPT billing codes mentioned.

        ### Target JSON Schema:
        {{
          "patient_analytics": {{
            "patient_info": {{
              "name": "string",
              "age": "integer",
              "gender": "string"
            }},
            "encounter_details": {{
              "date": "string",
              "provider": "string",
              "symptoms": ["string"]
            }},
            "clinical_data": {{
              "diagnosis": "string",
              "icd_10_code": "string",
              "medications": [
                {{
                  "name": "string",
                  "dosage": "string",
                  "frequency": "string"
                }}
              ]
            }},
            "billing_claims": {{
              "claim_id": "string",
              "cpt_codes": ["string"]
            }}
          }}
        }}

        Return ONLY valid JSON. Do not include any conversational text."""
    ),
    ("human", "Please extract the data from this clinical paragraph: {input_text}")
])

para = input("Give your clinical paragraph: ")
final_prompt = prompt.invoke({"input_text": para})
response = model.invoke(final_prompt)

# chain = prompt | llm
# response = chain.invoke({"input_text": para})


print(response.content)


parser = JsonOutputParser()
print("DEBUG: Parsed JSON: ", parser.parse(response.content))
print("-"*100)

# parser = StrOutputParser()
# print("DEBUG: Parsed String: ", parser.parse(response.content))
# print("-"*100)



Give your clinical paragraph: Patient Maria Garcia, a 62-year-old female, visited Dr. Smith on 2023-10-12. She reported fatigue and joint pain. Diagnosis: Rheumatoid Arthritis (M06.9). Prescribed Methotrexate 15mg weekly and Ibuprofen 400mg twice daily. Claim ID: MED12345. CPT: 99214, 85025
```json
{
  "patient_analytics": {
    "patient_info": {
      "name": "Maria Garcia",
      "age": 62,
      "gender": "Female"
    },
    "encounter_details": {
      "date": "2023-10-12",
      "provider": "Dr. Smith",
      "symptoms": [
        "fatigue",
        "joint pain"
      ]
    },
    "clinical_data": {
      "diagnosis": "Rheumatoid Arthritis",
      "icd_10_code": "M06.9",
      "medications": [
        {
          "name": "Methotrexate",
          "dosage": "15mg",
          "frequency": "weekly"
        },
        {
          "name": "Ibuprofen",
          "dosage": "400mg",
          "frequency": "twice daily"
        }
      ]
    },
    "billing_claims": {
      "claim_id": "ME